<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/GEMMA_13TASK_TOPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Sun Aug 23 00:34:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             58W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## SETUP

In [ ]:
!pip install scikit-fuzzy -q

# Install Hugging Face libraries
!pip install  --upgrade transformers datasets accelerate evaluate bitsandbytes --quiet

!pip install --upgrade optimum -q

!pip install textblob -q

from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

!pip install vllm==0.19.1 -q

!pip install unsloth -q

!pip install transformers==5.7.0 vllm -q

In [3]:
!pip show transformers accelerate scikit-learn vllm torch unsloth bitsandbytes | egrep  "Name|Version"

Name: transformers
Version: 5.7.0
Name: accelerate
Version: 1.14.0
Name: scikit-learn
Version: 1.6.1
 Name: GCC runtime library
 Version 3.1, 31 March 2009
Name: vllm
Version: 0.19.1
Name: torch
Version: 2.10.0
Name: unsloth
Version: 2026.8.19
Name: bitsandbytes
Version: 0.50.1


In [4]:
# ----------------------------------------------------------------------------
# GEMMA-4-E4B - QUIET LOAD (SUPPRESSES UNSLOTH BANNER)
# ----------------------------------------------------------------------------

print("\n👁️ Loading Vision Model: Gemma-4-E4B...")

# Suppress Unsloth output during loading
import contextlib
import io
import torch

vision_model = None
vision_processor = None

try:
    # Redirect stdout/stderr to suppress Unsloth banner
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        from unsloth import FastVisionModel

        vision_model, vision_processor = FastVisionModel.from_pretrained(
            "frankmorales2020/gemma-4-e4b-unesco-optimized",
            load_in_4bit=True,
            dtype=torch.bfloat16,
            device_map="auto",
        )
        FastVisionModel.for_inference(vision_model)

    print("✅ Gemma Loaded (Unsloth)")

except Exception as e:
    print(f"⚠️ Unsloth failed: {e}")
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer

        vision_model = AutoModelForCausalLM.from_pretrained(
            "frankmorales2020/gemma-4-e4b-unesco-optimized",
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True
        )
        vision_processor = AutoTokenizer.from_pretrained(
            "frankmorales2020/gemma-4-e4b-unesco-optimized",
            trust_remote_code=True
        )
        print("✅ Gemma Loaded (Transformers)")
    except Exception as e2:
        print(f"⚠️ Gemma failed: {e2}")
        vision_model = None
        vision_processor = None


👁️ Loading Vision Model: Gemma-4-E4B...


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

✅ Gemma Loaded (Unsloth)


In [ ]:
!pip install codecarbon -q

In [6]:
#!/usr/bin/env python3
import sys
import os
import contextlib

# ===== KILL ALL STDERR OUTPUT - THIS 100% SILENCES EVERYTHING =====
sys.stderr = open(os.devnull, 'w')

# ===== NOW IMPORT EVERYTHING =====
import gc, json, random, subprocess, warnings
import torch
import numpy as np
import psutil
import nltk
import requests
import time
from io import BytesIO
from PIL import Image
from codecarbon import EmissionsTracker

# ===== SUPPRESS ALL WARNINGS =====
warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["BITSANDBYTES_NOWELCOME"] = "1"

# Try unsloth, fallback to transformers
try:
    from unsloth import FastVisionModel
    USING_UNSLOTH = True
except:
    from transformers import AutoModelForVision2Seq, AutoProcessor
    USING_UNSLOTH = False

nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# ===== SILENCE STDOUT (suppresses bitsandbytes "Skipping..." spam) =====
@contextlib.contextmanager
def suppress_stdout():
    with open(os.devnull, 'w') as devnull:
        old_stdout = sys.stdout
        sys.stdout = devnull
        try:
            yield
        finally:
            sys.stdout = old_stdout

def set_reproducibility(seed=123):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"🔐 Determinism Locked | Seed: {seed}")

def global_memory_purge():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()

def get_ram_gb():
    return psutil.Process().memory_info().rss / (1024**3)

def get_vram_gb():
    return torch.cuda.memory_allocated() / (1024**3) if torch.cuda.is_available() else 0

def get_gpu_power_watts():
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=power.draw', '--format=csv,noheader,nounits'],
            capture_output=True, text=True
        )
        return float(result.stdout.strip().split('\n')[0])
    except:
        return 250.0

def convert_to_serializable(obj):
    if isinstance(obj, np.floating):  return float(obj)
    if isinstance(obj, np.integer):   return int(obj)
    if isinstance(obj, np.bool_):     return bool(obj)
    if isinstance(obj, np.ndarray):   return obj.tolist()
    if isinstance(obj, dict):         return {k: convert_to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):         return [convert_to_serializable(i) for i in obj]
    return obj

class QualityMetrics:
    def calculate_similarity(self, generated, image_name):
        generated = generated.lower().strip()
        if image_name == "Turing Award Winners":
            ai_godfathers = {
                "bengio": ["bengio", "yoshua"],
                "hinton": ["hinton", "geoffrey"],
                "lecun":  ["lecun",  "yann"]
            }
            names_found = sum(
                1 for variants in ai_godfathers.values()
                if any(v in generated for v in variants)
            )
            concepts = {
                "three":     ["three", "3"],
                "headshots": ["headshots", "portraits", "photos"],
                "ai":        ["artificial intelligence", "ai", "deep learning"],
                "award":     ["turing", "award", "prize"]
            }
            concept_score = sum(
                1 for synonyms in concepts.values()
                if any(s in generated for s in synonyms)
            ) / len(concepts)
            score = (names_found / 3.0 * 0.8) + (concept_score * 0.2)
            if names_found == 3:
                score = max(score, 0.95)
            return float(min(score, 1.0))
        if image_name == "Bee on Flower":
            key_elements = {
                "bee":    ["bee", "honeybee", "bumblebee"],
                "flower": ["flower", "blossom", "bloom", "cosmos", "petal"],
                "pink":   ["pink", "vibrant", "magenta", "purple"]
            }
            score = sum(
                1 for synonyms in key_elements.values()
                if any(s in generated for s in synonyms)
            ) / len(key_elements)
            if "bee" in generated and ("flower" in generated or "bloom" in generated):
                score = max(score, 0.85)
            return float(min(score, 1.0))
        if image_name == "Wisconsin Boardwalk":
            key_elements = {
                "boardwalk": ["boardwalk", "walkway", "path", "wooden"],
                "nature":    ["field", "grass", "green", "landscape"],
                "sky":       ["sky", "clouds", "horizon"]
            }
            score = sum(
                1 for synonyms in key_elements.values()
                if any(s in generated for s in synonyms)
            ) / len(key_elements)
            if ("boardwalk" in generated or "wooden" in generated) and \
               ("field" in generated or "grass" in generated):
                score = max(score, 0.85)
            return float(min(score, 1.0))
        return 0.0

test_images = [
    {"name": "Bee on Flower",        "url": "https://raw.githubusercontent.com/frank-morales2020/UNESCO2026/main/images/bee_on_flower.jpg"},
    {"name": "Wisconsin Boardwalk",  "url": "https://raw.githubusercontent.com/frank-morales2020/UNESCO2026/main/images/wisconsin_boardwalk.jpg"},
    {"name": "Turing Award Winners", "url": "https://raw.githubusercontent.com/frank-morales2020/UNESCO2026/main/images/turing_award_winners.jpg"},
]

def load_image(item):
    try:
        r = requests.get(item["url"], headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
        r.raise_for_status()
        return Image.open(BytesIO(r.content)).convert("RGB")
    except Exception as e:
        print(f"  ⚠️ Could not load {item['name']}: {e}")
        return None

def build_inputs(model, processor, image, prompt):
    messages = [{"role": "user", "content": [
        {"type": "image"}, {"type": "text", "text": prompt}
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return processor(text=text, images=[image], return_tensors="pt").to(model.device)

# ===== MAIN EVALUATION =====
print("=" * 80)
print("GEMMA 4 E4B — EVALUATION FROM HF")
print("=" * 80)

MODEL_PATH = "frankmorales2020/gemma-4-e4b-unesco-optimized"

set_reproducibility(123)
os.makedirs("./carbon_emissions", exist_ok=True)
global_memory_purge()

print(f"\n📦 Loading model from: {MODEL_PATH}")

# ===== LOAD MODEL — stdout suppressed to silence bitsandbytes "Skipping..." spam =====
if USING_UNSLOTH:
    with suppress_stdout():
        model, processor = FastVisionModel.from_pretrained(
            MODEL_PATH,
            load_in_4bit=True,
            dtype=torch.bfloat16,
            device_map="auto",
        )
        FastVisionModel.for_inference(model)
    print("✓ Loaded with Unsloth")
else:
    with suppress_stdout():
        model = AutoModelForVision2Seq.from_pretrained(
            MODEL_PATH,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True
        )
        processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)
    print("✓ Loaded with Transformers")

global_memory_purge()
print(f"✓ Loaded — VRAM: {get_vram_gb():.2f} GB | RAM: {get_ram_gb():.2f} GB")

# Run benchmark
print("\n" + "=" * 80)
print("🔬 RUNNING UNESCO BENCHMARK")
print("=" * 80)

qm = QualityMetrics()
results = []
tracker = EmissionsTracker(
    project_name="gemma4_unesco_eval",
    output_dir="./carbon_emissions",
    save_to_file=True,
    log_level="ERROR"
)
tracker.start()

for idx, item in enumerate(test_images, 1):
    print(f"\n{'='*60}\n📸 [{idx}/3] {item['name']}\n{'='*60}")
    image = load_image(item)
    if image is None:
        results.append({"name": item['name'], "quality_score": 0.0, "error": True})
        continue
    print("  ✅ Image loaded")

    inputs = build_inputs(model, processor, image, "Describe this image.")
    global_memory_purge()
    power_start = get_gpu_power_watts()
    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            use_cache=True,
            do_sample=False,
            temperature=1.0,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    generation_time = time.time() - start_time
    cpu_usage = psutil.cpu_percent(interval=0.1)
    ram_after = get_ram_gb()
    vram_after = get_vram_gb()
    power_end = get_gpu_power_watts()
    avg_power = (power_start + power_end) / 2

    input_len = inputs["input_ids"].shape[1]
    generated = processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    for prefix in ["Describe this image.", "model", "assistant"]:
        if generated.lower().startswith(prefix.lower()):
            generated = generated[len(prefix):].strip()
    if not generated:
        generated = "No description generated"

    quality_score = qm.calculate_similarity(generated, item['name'])
    output_words = len(generated.split())
    rtf = generation_time / max(output_words, 1)
    throughput = output_words / generation_time if generation_time > 0 else 0
    energy_joules = avg_power * generation_time
    energy_kwh = energy_joules / (1000 * 3600)
    peak_vram = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0

    result = {
        "name": item['name'], "generated": generated[:300],
        "quality_score": float(quality_score), "generation_time": float(generation_time),
        "rtf": float(rtf), "throughput": float(throughput), "output_words": int(output_words),
        "ram_gb": float(ram_after), "vram_gb": float(vram_after), "peak_vram_gb": float(peak_vram),
        "cpu_usage": float(cpu_usage), "energy_joules": float(energy_joules),
        "energy_kwh": float(energy_kwh), "avg_power_watts": float(avg_power)
    }
    results.append(result)

    print(f"\n  📝 Generated: {generated[:200]}...")
    print(f"  ⏱️  Time: {generation_time:.2f}s | RTF: {rtf:.4f} s/word | Words: {output_words}")
    print(f"  🚀 Throughput: {throughput:.1f} words/sec")
    print(f"  🔋 Energy: {energy_joules:.2f} J | {energy_kwh:.6f} kWh | Power: {avg_power:.1f}W")
    print(f"  💻 CPU: {cpu_usage:.1f}% | RAM: {ram_after:.2f} GB | VRAM: {vram_after:.2f} GB")
    print(f"  🎯 SEMANTIC SCORE: {quality_score:.3f}")

    if item['name'] == "Turing Award Winners":
        gen_lower = generated.lower()
        names = []
        if "bengio" in gen_lower or "yoshua" in gen_lower: names.append("Yoshua Bengio")
        if "hinton" in gen_lower or "geoffrey" in gen_lower: names.append("Geoffrey Hinton")
        if "lecun" in gen_lower or "yann" in gen_lower: names.append("Yann LeCun")
        if names:
            print(f"  🎯 AI GODFATHERS IDENTIFIED: {', '.join(names)}")

    global_memory_purge()

emissions_data = tracker.stop()
total_co2 = emissions_data if isinstance(emissions_data, float) else 0.0

# Results
print("\n" + "=" * 80)
print("📊 EVALUATION RESULTS — GEMMA 4 E4B (Loaded from HDD)")
print("=" * 80)

valid_results = [r for r in results if not r.get("error", False)]

if valid_results:
    avg_quality = float(np.mean([r['quality_score'] for r in valid_results]))
    avg_rtf = float(np.mean([r['rtf'] for r in valid_results]))
    avg_ram = float(np.mean([r['ram_gb'] for r in valid_results]))
    avg_vram = float(np.mean([r['vram_gb'] for r in valid_results]))
    avg_cpu = float(np.mean([r['cpu_usage'] for r in valid_results]))
    total_energy = float(np.sum([r['energy_joules'] for r in valid_results]))
    avg_throughput = float(np.mean([r['throughput'] for r in valid_results]))
    ram_pass = avg_ram < 4.0
    rtf_pass = avg_rtf < 1.0
    quality_pass = avg_quality > 0.8

    print(f"\n  Average RAM:           {avg_ram:.2f} GB")
    print(f"  Average VRAM:          {avg_vram:.2f} GB")
    print(f"  Average CPU Load:      {avg_cpu:.1f} %")
    print(f"  Average RTF:           {avg_rtf:.4f} sec/word")
    print(f"  Average Throughput:    {avg_throughput:.1f} words/sec")
    print(f"  Total Energy:          {total_energy:.2f} J")
    print(f"  Total CO2e:            {total_co2:.6f} kg")
    print(f"  Average Quality Score: {avg_quality:.3f}")
    print(f"\n🔍 CHALLENGE TARGETS:")
    print(f"  RAM < 4GB:    {'✅ PASS' if ram_pass else '❌ FAIL'} ({avg_ram:.2f} GB)")
    print(f"  RTF < 1.0:    {'✅ PASS' if rtf_pass else '❌ FAIL'} ({avg_rtf:.4f})")
    print(f"  Quality >80%: {'✅ PASS' if quality_pass else '❌ FAIL'} ({avg_quality:.3f})")

    if ram_pass and rtf_pass and quality_pass:
        print("\n🎉 ALL CHALLENGE TARGETS ACHIEVED! 🎉")
    else:
        print("\n⚠️ Some targets not yet achieved.")
else:
    print("\n❌ No successful validations")

# Save results
print("\n" + "=" * 80)
print("💾 SAVING EVALUATION RESULTS")
print("=" * 80)

EVAL_DIR = "evaluation_results"
os.makedirs(EVAL_DIR, exist_ok=True)

evaluation = {
    "model": "google/gemma-4-E4B-it",
    "model_path": MODEL_PATH,
    "evaluation_date": time.strftime("%Y-%m-%d %H:%M:%S"),
    "metrics": {
        "average_quality_score": avg_quality if valid_results else 0,
        "average_rtf_sec_per_word": avg_rtf if valid_results else 0,
        "average_throughput_words_per_sec": avg_throughput if valid_results else 0,
        "average_ram_gb": avg_ram if valid_results else 0,
        "average_vram_gb": avg_vram if valid_results else 0,
        "average_cpu_percent": avg_cpu if valid_results else 0,
        "total_energy_joules": total_energy,
        "total_co2_kg": float(total_co2),
    },
    "individual_results": valid_results,
    "challenge_targets_met": {
        "ram_under_4gb": bool(ram_pass) if valid_results else False,
        "rtf_under_1": bool(rtf_pass) if valid_results else False,
        "quality_over_80": bool(quality_pass) if valid_results else False,
    }
}

with open(os.path.join(EVAL_DIR, "evaluation_metrics.json"), "w") as f:
    json.dump(convert_to_serializable(evaluation), f, indent=2)

print(f"\n✅ Evaluation saved to: {EVAL_DIR}/evaluation_metrics.json")
print("\n" + "=" * 80)
print("✅ EVALUATION COMPLETE")
print("=" * 80)

GEMMA 4 E4B — EVALUATION FROM HF
🔐 Determinism Locked | Seed: 123

📦 Loading model from: frankmorales2020/gemma-4-e4b-unesco-optimized


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

✓ Loaded with Unsloth
✓ Loaded — VRAM: 20.14 GB | RAM: 2.22 GB

🔬 RUNNING UNESCO BENCHMARK

📸 [1/3] Bee on Flower
  ✅ Image loaded

  📝 Generated: This is a close-up photograph of a vibrant pink flower, likely a type of cosmos, in a garden setting.

**Key elements in the image:**

*   **The Flower:** The central focus is a large, beautiful pink ...
  ⏱️  Time: 54.98s | RTF: 0.4823 s/word | Words: 114
  🚀 Throughput: 2.1 words/sec
  🔋 Energy: 3762.13 J | 0.001045 kWh | Power: 68.4W
  💻 CPU: 0.0% | RAM: 3.09 GB | VRAM: 20.14 GB
  🎯 SEMANTIC SCORE: 1.000

📸 [2/3] Wisconsin Boardwalk
  ✅ Image loaded

  📝 Generated: This is a vibrant, wide-angle photograph of a natural landscape, dominated by a long, wooden boardwalk cutting through a lush, green field under a bright, expansive sky.

**Foreground and Midground:**...
  ⏱️  Time: 21.51s | RTF: 0.1886 s/word | Words: 114
  🚀 Throughput: 5.3 words/sec
  🔋 Energy: 1573.12 J | 0.000437 kWh | Power: 73.2W
  💻 CPU: 0.0% | RAM: 3.15 GB | VRAM: 20.1

## TOPO-GEMMA4-STL

In [ ]:
# ============================================================================
# TOPO-2026: 13 TASKS EXTENDED (OPTIMIZED & CLEANED SCRIPT)
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import gc
import random
import os
import contextlib
import io
from sklearn.metrics import accuracy_score
from tqdm import tqdm
from transformers import AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("🔬 TOPO-2026: 13 TASKS EXTENDED (OPTIMIZED)")
print("   5 RUNS - MULTI-TASK MASTER")
print("="*80)

# ============================================================================
# 1. CONFIGURATION
# ============================================================================
SEED = 123
N_RUNS = 5
BATCH_SIZE = 8
MAX_EPOCHS = 10
PATIENCE = 2
PRIME_LIMIT = 13
MAX_LEN = 64
NUM_TASKS = 13
BOUNDARY_LAYER = 24  # Exact boundary layer targeting

MODEL_NAME = "frankmorales2020/gemma-4-e4b-unesco-optimized"

# FIXED LR GRID
LR_GRID = [
    (5e-3, 1e-3),
    (1e-3, 5e-4),
    (5e-3, 5e-3),
    (2e-3, 1e-3),
    (1e-3, 1e-3),
]

PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]
SAFETY_CONSTANT = 1.0 - np.prod([1.0 - (p ** -0.5) for p in PRIME_ANCHORS])

print(f"\n📋 Configuration:")
print(f"   Model: {MODEL_NAME}")
print(f"   Runs: {N_RUNS}")
print(f"   Tasks: {NUM_TASKS}")
print(f"   Epochs: {MAX_EPOCHS}")
print(f"   Boundary Layer: {BOUNDARY_LAYER}")
print(f"   Prime Anchors: {PRIME_ANCHORS}")

# ============================================================================
# 2. LOAD VISION MODEL
# ============================================================================
print(f"\n👁️ Loading Vision Model: Gemma-4-E4B...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"   Device: {device}")

vision_model = None
vision_processor = None

try:
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        from unsloth import FastVisionModel

        vision_model, vision_processor = FastVisionModel.from_pretrained(
            MODEL_NAME,
            load_in_4bit=True,
            dtype=torch.bfloat16,
            device_map="auto",
        )
        FastVisionModel.for_inference(vision_model)

    print("✅ Gemma Loaded (Unsloth)")

except Exception as e:
    print(f"⚠️ Unsloth failed: {e}")
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer

        vision_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True
        )
        vision_processor = AutoTokenizer.from_pretrained(
            MODEL_NAME,
            trust_remote_code=True
        )
        print("✅ Gemma Loaded (Transformers)")
    except Exception as e2:
        print(f"⚠️ Gemma failed: {e2}")
        vision_model = None
        vision_processor = None

# ============================================================================
# 3. GET TOKENIZER
# ============================================================================
if vision_processor is not None:
    if hasattr(vision_processor, 'tokenizer'):
        tokenizer = vision_processor.tokenizer
    else:
        tokenizer = vision_processor
else:
    tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b", trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

hidden_size = 2560

print(f"\n   ✅ Model ready!")
print(f"   Hidden Size: {hidden_size}")
print(f"   Vocab Size: {len(tokenizer)}")

if vision_model is not None:
    vision_model = vision_model.to(device)
    for param in vision_model.parameters():
        param.requires_grad = False

# ============================================================================
# 4. DATASET - STL-10
# ============================================================================
STL_CLASSES = {
    0: 'airplane', 1: 'bird', 2: 'car', 3: 'cat', 4: 'deer',
    5: 'dog', 6: 'horse', 7: 'monkey', 8: 'ship', 9: 'truck'
}

print(f"\n📌 TASKS:")
print(f"   A: Animal vs Vehicle")
print(f"   B: Natural vs Man-Made")
print(f"   C: Living vs Non-Living")
print(f"   D: Large vs Small")
print(f"   E: Ground vs Air/Water")
print(f"   F: Domestic vs Wild")
print(f"   G: Mammal vs Non-Mammal")
print(f"   H: Flying vs Non-Flying")
print(f"   I: Fast vs Slow")
print(f"   J: Urban vs Rural")
print(f"   K: Predator vs Prey")
print(f"   L: Nocturnal vs Diurnal")
print(f"   M: Domesticated vs Wild Animals")

# ============================================================================
# 5. LOAD STL-10
# ============================================================================
print(f"\n📚 LOADING STL-10")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.STL10(
    root='./data', split='train', download=True, transform=transform
)
testset = torchvision.datasets.STL10(
    root='./data', split='test', download=True, transform=transform
)

print(f"   Training set: {len(trainset):,} samples")
print(f"   Test set: {len(testset):,} samples")

# ============================================================================
# 6. 13 TASK DEFINITIONS
# ============================================================================
def get_class_label(cls, task):
    if cls in task['class0']:
        return 0
    else:
        return 1

TASKS_13 = {
    'A': {
        'name': 'Animal vs Vehicle',
        'class0': [1, 3, 4, 5, 6, 7],
        'class1': [0, 2, 8, 9],
        'label0_text': ['animal', 'living creature', 'wild animal'],
        'label1_text': ['vehicle', 'machine', 'transportation']
    },
    'B': {
        'name': 'Natural vs Man-Made',
        'class0': [1, 3, 4, 5, 6, 7],
        'class1': [0, 2, 8, 9],
        'label0_text': ['natural', 'organic', 'from nature'],
        'label1_text': ['man-made', 'artificial', 'human-built']
    },
    'C': {
        'name': 'Living vs Non-Living',
        'class0': [1, 3, 4, 5, 6, 7],
        'class1': [0, 2, 8, 9],
        'label0_text': ['living', 'alive', 'breathing'],
        'label1_text': ['non-living', 'inanimate', 'not alive']
    },
    'D': {
        'name': 'Large vs Small',
        'class0': [0, 2, 6, 8, 9],
        'class1': [1, 3, 4, 5, 7],
        'label0_text': ['large', 'big', 'large-sized'],
        'label1_text': ['small', 'tiny', 'small-sized']
    },
    'E': {
        'name': 'Ground vs Air/Water',
        'class0': [2, 3, 5, 6, 7],
        'class1': [0, 1, 4, 8, 9],
        'label0_text': ['ground', 'land-based', 'terrestrial'],
        'label1_text': ['air or water', 'non-terrestrial', 'flying/swimming']
    },
    'F': {
        'name': 'Domestic vs Wild',
        'class0': [2, 3, 5],
        'class1': [1, 4, 6, 7],
        'label0_text': ['domestic', 'tame', 'pet'],
        'label1_text': ['wild', 'untamed', 'savage']
    },
    'G': {
        'name': 'Mammal vs Non-Mammal',
        'class0': [3, 5, 6, 7],
        'class1': [0, 1, 2, 4, 8, 9],
        'label0_text': ['mammal', 'warm-blooded', 'fur-bearing'],
        'label1_text': ['non-mammal', 'cold-blooded', 'feathered/metal']
    },
    'H': {
        'name': 'Flying vs Non-Flying',
        'class0': [0, 1],
        'class1': [2, 3, 4, 5, 6, 7, 8, 9],
        'label0_text': ['flying', 'can fly', 'airborne'],
        'label1_text': ['non-flying', 'ground-based', 'earthbound']
    },
    'I': {
        'name': 'Fast vs Slow',
        'class0': [0, 2, 6, 8, 9],
        'class1': [1, 3, 4, 5, 7],
        'label0_text': ['fast-moving', 'quick', 'rapid'],
        'label1_text': ['slow-moving', 'slow', 'lethargic']
    },
    'J': {
        'name': 'Urban vs Rural',
        'class0': [0, 2, 8, 9],
        'class1': [1, 3, 4, 5, 6, 7],
        'label0_text': ['urban', 'city', 'man-made environment'],
        'label1_text': ['rural', 'countryside', 'natural environment']
    },
    'K': {
        'name': 'Predator vs Prey',
        'class0': [3, 5, 7],
        'class1': [1, 4, 6],
        'label0_text': ['predator', 'hunter', 'carnivore'],
        'label1_text': ['prey', 'herbivore', 'hunted']
    },
    'L': {
        'name': 'Nocturnal vs Diurnal',
        'class0': [3, 5, 7],
        'class1': [1, 4, 6],
        'label0_text': ['nocturnal', 'night-active', 'night'],
        'label1_text': ['diurnal', 'day-active', 'day']
    },
    'M': {
        'name': 'Domesticated vs Wild Animals',
        'class0': [3, 5],
        'class1': [1, 4, 6, 7],
        'label0_text': ['domesticated', 'pet', 'tame animal'],
        'label1_text': ['wild animal', 'untamed', 'free']
    },
}

TASK_ORDER = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M']

# ============================================================================
# 7. CREATE DATASETS
# ============================================================================
def create_vision_text(label, task_type):
    class_name = STL_CLASSES[label]
    task = TASKS_13[task_type]

    if label in task['class0']:
        prefixes = [f"A {class_name} {t}" for t in task['label0_text']]
        prefixes += [f"A {t} {class_name}" for t in task['label0_text']]
    else:
        prefixes = [f"A {class_name} {t}" for t in task['label1_text']]
        prefixes += [f"A {t} {class_name}" for t in task['label1_text']]

    return random.choice(prefixes)

def create_stl_text_dataset(dataset, class_list, num_samples, task_type):
    random.seed(SEED)
    texts, labels = [], []
    samples_per_class = num_samples // len(class_list)

    for cls in class_list:
        indices = [i for i, (_, label) in enumerate(dataset) if label == cls]
        available = min(len(indices), samples_per_class * 3)
        selected = random.sample(indices, available)
        for idx in selected:
            texts.append(create_vision_text(cls, task_type))
            labels.append(get_class_label(cls, TASKS_13[task_type]))

    return texts, labels

num_samples = 2000

task_loaders = {}
test_loaders = {}

print(f"\n📚 Creating 13 task datasets...")
for task_id in TASK_ORDER:
    task = TASKS_13[task_id]
    class_list = task['class0'] + task['class1']

    print(f"   Task {task_id}: {task['name']}")

    texts, labels = create_stl_text_dataset(trainset, class_list, num_samples, task_id)
    tokens = tokenizer(texts, max_length=MAX_LEN, padding='max_length', truncation=True, return_tensors='pt')

    dataset = torch.utils.data.TensorDataset(
        tokens.input_ids,
        tokens.attention_mask,
        torch.tensor(labels, dtype=torch.long)
    )

    loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    task_loaders[task_id] = loader

    test_texts, test_labels = create_stl_text_dataset(testset, class_list, 400, task_id)
    test_tokens = tokenizer(test_texts, max_length=MAX_LEN, padding='max_length', truncation=True, return_tensors='pt')

    test_dataset = torch.utils.data.TensorDataset(
        test_tokens.input_ids,
        test_tokens.attention_mask,
        torch.tensor(test_labels, dtype=torch.long)
    )

    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loaders[task_id] = test_loader

    print(f"      Training: {len(texts)} samples, Test: {len(test_texts)} samples")

# ============================================================================
# 8. CLASSIFIER MODEL - 13 HEADS (WITH BOUNDARY LAYER 24)
# ============================================================================
class GemmaVisionClassifier(nn.Module):
    def __init__(self, vision_model, hidden_size=2560, boundary_layer=BOUNDARY_LAYER):
        super().__init__()
        self.vision_model = vision_model
        self.hidden_size = hidden_size
        self.boundary_layer = boundary_layer

        for task_id in TASK_ORDER:
            setattr(self, f'classifier_{task_id}', nn.Linear(hidden_size, 2))

        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs = self.vision_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )

        if hasattr(outputs, 'hidden_states') and outputs.hidden_states is not None:
            if len(outputs.hidden_states) > self.boundary_layer:
                hidden_states = outputs.hidden_states[self.boundary_layer]
            else:
                hidden_states = outputs.hidden_states[-1]
        else:
            hidden_states = outputs.last_hidden_state

        hidden_states = hidden_states.float()

        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)
        else:
            pooled = hidden_states.mean(dim=1)

        head = getattr(self, f'classifier_{self.current_task}')
        return head(pooled)

    def switch_task(self, task):
        assert task in TASK_ORDER
        self.current_task = task

    def freeze_previous_heads(self, task):
        task_idx = TASK_ORDER.index(task)
        for i in range(task_idx):
            prev_task = TASK_ORDER[i]
            head = getattr(self, f'classifier_{prev_task}')
            for param in head.parameters():
                param.requires_grad = False

# ============================================================================
# 9. TOPOLOGICAL GOVERNOR (WITH BOUNDARY LAYER 24)
# ============================================================================
class TopologicalGovernor:
    def __init__(self, model: nn.Module, boundary_layer=BOUNDARY_LAYER):
        self.model = model
        self.boundary_layer = boundary_layer
        self.reference_anchors = {}
        self.safety_constant = SAFETY_CONSTANT
        self.snapshot = {}
        self._register_topo_anchors()

    def _register_topo_anchors(self):
        print(f"Initializing TOPO-2026 Topological Governor anchor snapshots (Boundary Layer: {self.boundary_layer})...")
        count = 0
        with torch.no_grad():
            for name, param in self.model.named_parameters():
                if (param.is_floating_point() or param.is_complex()) and param.ndim >= 1:
                    if f"layers.{self.boundary_layer}" in name or f"blocks.{self.boundary_layer}" in name or any(f"layer.{b}" in name for b in [23, 24, 25]):
                        snapshot = {}
                        for p in PRIME_ANCHORS:
                            if p < param.shape[0]:
                                snapshot[p] = param.data[p].clone()
                        if snapshot:
                            self.reference_anchors[name] = snapshot
                            count += 1
        print(f"Topological Governor successfully locked prime reference anchors across {count} tensors at Boundary Layer {self.boundary_layer}.")

    def take_snapshot(self):
        embed_layer = self.model.vision_model.get_input_embeddings()
        vocab_size = embed_layer.weight.shape[0]
        anchor_indices = [p for p in PRIME_ANCHORS if p < vocab_size]
        self.snapshot = {idx: embed_layer.weight[idx].detach().clone().float() for idx in anchor_indices}
        self._register_topo_anchors()

    @torch.no_grad()
    def enforce_anchors(self):
        if not self.reference_anchors:
            return
        with torch.no_grad():
            for name, param in self.model.named_parameters():
                if name in self.reference_anchors:
                    dtype = param.dtype
                    for p, val in self.reference_anchors[name].items():
                        if p < param.shape[0]:
                            param.data[p].copy_(val.to(dtype=dtype))

    @torch.no_grad()
    def zero_anchor_gradients(self):
        with torch.no_grad():
            for name, param in self.model.named_parameters():
                if param.requires_grad and param.ndim >= 1 and param.grad is not None:
                    if name in self.reference_anchors:
                        for p in self.reference_anchors[name].keys():
                            if p < param.grad.shape[0]:
                                param.grad[p] = 0.0

    def verify_integrity(self, atol: float = 1e-5) -> bool:
        if not self.reference_anchors:
            return True
        with torch.no_grad():
            for name, param in self.model.named_parameters():
                if name in self.reference_anchors:
                    for p, val in self.reference_anchors[name].items():
                        if p < param.shape[0]:
                            if not torch.allclose(param.data[p].float(), val.float(), atol=atol):
                                return False
        return True

# ============================================================================
# 10. TRAINING FUNCTIONS
# ============================================================================
def train_task(task_label, model, loader, governor, lr_embed, lr_cls, max_epochs, patience):
    model.switch_task(task_label)
    model.train()

    head = getattr(model, f'classifier_{task_label}')
    embed_layer = model.vision_model.get_input_embeddings()

    optimizer = torch.optim.AdamW([
        {'params': embed_layer.parameters(), 'lr': lr_embed, 'weight_decay': 1e-4},
        {'params': head.parameters(), 'lr': lr_cls, 'weight_decay': 1e-4},
    ])

    best_acc = 0.0
    patience_counter = 0
    best_classifier_state = None

    for epoch in range(max_epochs):
        epoch_loss = 0
        num_batches = 0

        for input_ids, attention_mask, labels in tqdm(loader, desc=f"   Epoch {epoch+1}/{max_epochs}", leave=False):
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss = F.cross_entropy(logits, labels)
            loss.backward()

            if governor:
                governor.zero_anchor_gradients()

            torch.nn.utils.clip_grad_norm_(embed_layer.parameters(), max_norm=1.0)
            optimizer.step()

            if governor:
                governor.enforce_anchors()

            epoch_loss += loss.item()
            num_batches += 1

        avg_loss = epoch_loss / num_batches
        val_acc = evaluate_model(model, test_loaders[task_label], task_label)

        print(f"   Epoch {epoch+1}/{max_epochs}: Loss={avg_loss:.4f}, Val Acc={val_acc*100:.2f}%")

        if val_acc > best_acc:
            best_acc = val_acc
            patience_counter = 0
            # FIXED: Only save the state dict for the current task head to prevent checkpoint state leakage across tasks
            best_classifier_state = head.state_dict()
            print(f"     ✅ New best: {best_acc*100:.2f}%")
        else:
            patience_counter += 1
            print(f"     ⏳ No improvement ({patience_counter}/{patience})")

        if patience_counter >= patience and epoch > 1:
            print(f"     🛑 EARLY STOPPING at epoch {epoch+1}")
            if best_classifier_state is not None:
                head.load_state_dict(best_classifier_state)
                model.to(device)
            break

    # Ensure best state is loaded even if early stopping wasn't triggered
    if best_classifier_state is not None:
        head.load_state_dict(best_classifier_state)
        model.to(device)

@torch.no_grad()
def evaluate_model(model, loader, task):
    model.switch_task(task)
    model.eval()

    all_preds, all_labels = [], []
    for input_ids, attention_mask, labels in loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)
        logits = model(input_ids, attention_mask)
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return accuracy_score(all_labels, all_preds)

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

def flush_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

def cleanup(*objects):
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()

# ============================================================================
# 11. MAIN TRAINING LOOP - 5 RUNS
# ============================================================================
print(f"\n" + "="*80)
print(f"🚀 STARTING 5-RUN TRAINING (13 TASKS)")
print("="*80)

all_results = []
best_run = None
global_best_model_state = None
global_best_avg_acc = 0.0

for run_id in range(N_RUNS):
    set_seed(SEED + run_id)
    lr_embed, lr_cls = LR_GRID[run_id]

    print(f"\n  {'═'*80}")
    print(f"  RUN {run_id + 1}/{N_RUNS}  |  lr_embed={lr_embed:.0e}  lr_cls={lr_cls:.0e}")
    print(f"  {'═'*80}")

    model = GemmaVisionClassifier(vision_model, hidden_size, boundary_layer=BOUNDARY_LAYER).to(device)
    embed_layer = model.vision_model.get_input_embeddings()
    embed_layer.weight.requires_grad = True

    print(f"\n  [ZERO-SHOT] Evaluating tasks...")
    zero_accs = {}
    for task_id in TASK_ORDER[:5]:
        zero_accs[task_id] = evaluate_model(model, test_loaders[task_id], task_id)
    zero_str = ", ".join([f"{k}={zero_accs[k]*100:.2f}%" for k in zero_accs])
    print(f"    Zero-shot (first 5): {zero_str}")

    governor = None
    task_peak_accs = {t: 0.0 for t in TASK_ORDER}

    for task_idx, task_id in enumerate(TASK_ORDER):
        print(f"\n  📚 TASK {task_id}: {TASKS_13[task_id]['name']}")

        if task_idx == 0:
            governor = TopologicalGovernor(model, boundary_layer=BOUNDARY_LAYER)
            governor.take_snapshot()
            print(f"  🔒 Anchored prime embeddings at Layer {BOUNDARY_LAYER}")
            print(f"  🔒 Safety Constant Λ: {governor.safety_constant:.10f}")
        else:
            model.freeze_previous_heads(task_id)

        train_task(task_id, model, task_loaders[task_id], governor,
                   lr_embed, lr_cls, MAX_EPOCHS, PATIENCE)

        for past_idx in range(task_idx + 1):
            past_task = TASK_ORDER[past_idx]
            curr_acc = evaluate_model(model, test_loaders[past_task], past_task)
            if curr_acc > task_peak_accs[past_task]:
                task_peak_accs[past_task] = curr_acc

        assert governor.verify_integrity(), f"❌ Topological integrity violated at Task {task_id}!"

    print(f"\n  📊 FINAL ACCURACIES & FORGETTING SCORE (all 13 tasks):")
    final_accs = {}
    task_forgetting = {}

    for task_id in TASK_ORDER:
        acc = evaluate_model(model, test_loaders[task_id], task_id)
        final_accs[task_id] = acc
        peak = task_peak_accs[task_id]
        fgt = max(0.0, peak - acc)
        task_forgetting[task_id] = fgt
        print(f"    Task {task_id} ({TASKS_13[task_id]['name'][:20]:20}): Acc={acc*100:.2f}% | Peak={peak*100:.2f}% | FGT={fgt*100:.2f}%")

    global_fgt = np.mean(list(task_forgetting.values()))
    print(f"\n  📉 Global Average Forgetting Score (FGT) for Run {run_id + 1}: {global_fgt*100:.4f}%")

    all_perfect = all(acc == 1.0 for acc in final_accs.values())
    if all_perfect:
        print(f"  🎉🎉🎉 ALL 13 TASKS AT 100%! 🎉🎉🎉")

    avg_acc = np.mean(list(final_accs.values()))
    if avg_acc > global_best_avg_acc:
        global_best_avg_acc = avg_acc
        global_best_model_state = {
            t: getattr(model, f'classifier_{t}').state_dict()
            for t in TASK_ORDER
        }
        best_run = run_id

    run_result = {
        'run_id': run_id,
        'lr_embed': lr_embed,
        'lr_cls': lr_cls,
        'all_perfect': all_perfect,
        'avg_accuracy': float(avg_acc * 100),
        'global_forgetting': float(global_fgt * 100),
        'final_accs': {k: float(v * 100) for k, v in final_accs.items()},
    }
    all_results.append(run_result)

    cleanup(model)
    flush_gpu()

# ============================================================================
# 12. SAVE EVERYTHING
# ============================================================================
print(f"\n" + "="*80)
print(f"💾 SAVING EVERYTHING TO LOCAL DISK")
print("="*80)

SAVE_DIR = "./topo_stl10_13tasks"
os.makedirs(SAVE_DIR, exist_ok=True)

embed_layer = vision_model.get_input_embeddings()
embed_w = embed_layer.weight.detach().cpu().float()

torch.save({
    'classifiers': global_best_model_state,
    'embed_tokens_weight': embed_w,
    'prime_anchors': PRIME_ANCHORS,
    'boundary_layer': BOUNDARY_LAYER,
    'safety_constant': SAFETY_CONSTANT,
    'hidden_size': hidden_size,
    'seed': SEED,
    'runs': N_RUNS,
    'task_order': TASK_ORDER,
    'task_definitions': TASKS_13,
    'best_run': best_run + 1 if best_run is not None else 0,
    'best_avg_acc': float(global_best_avg_acc),
}, f"{SAVE_DIR}/topo_trained_13tasks_gemma.pt")

print(f"   ✅ Saved: {SAVE_DIR}/topo_trained_13tasks_gemma.pt")
print("="*80)
print("🎉 COMPLETE! ALL FILES SAVED!")
print("="*80)

🔬 TOPO-2026: 13 TASKS EXTENDED (OPTIMIZED)
   5 RUNS - MULTI-TASK MASTER

📋 Configuration:
   Model: frankmorales2020/gemma-4-e4b-unesco-optimized
   Runs: 5
   Tasks: 13
   Epochs: 10
   Boundary Layer: 24
   Prime Anchors: [2, 3, 5, 7, 11, 13]

👁️ Loading Vision Model: Gemma-4-E4B...
   Device: cuda


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Gemma4ForConditionalGeneration LOAD REPORT from: frankmorales2020/gemma-4-e4b-unesco-optimized
Key                                                     | Status     |  | 
--------------------------------------------------------+------------+--+-
language_model.layers.{24...41}.self_attn.v_proj.weight | UNEXPECTED |  | 
language_model.layers.{24...41}.self_attn.k_norm.weight | UNEXPECTED |  | 
language_model.layers.{24...41}.self_attn.k_proj.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Gemma Loaded (Unsloth)

   ✅ Model ready!
   Hidden Size: 2560
   Vocab Size: 262144

📌 TASKS:
   A: Animal vs Vehicle
   B: Natural vs Man-Made
   C: Living vs Non-Living
   D: Large vs Small
   E: Ground vs Air/Water
   F: Domestic vs Wild
   G: Mammal vs Non-Mammal
   H: Flying vs Non-Flying
   I: Fast vs Slow
   J: Urban vs Rural
   K: Predator vs Prey
   L: Nocturnal vs Diurnal
   M: Domesticated vs Wild Animals

📚 LOADING STL-10


100%|██████████| 2.64G/2.64G [04:45<00:00, 9.26MB/s]


...

   Epoch 2/10: Loss=0.0000, Val Acc=100.00%
     ⏳ No improvement (1/2)


   Epoch 3/10: Loss=0.0000, Val Acc=100.00%
     ⏳ No improvement (2/2)
     🛑 EARLY STOPPING at epoch 3

  📚 TASK M: Domesticated vs Wild Animals


   Epoch 1/10: Loss=0.0045, Val Acc=100.00%
     ✅ New best: 100.00%


   Epoch 2/10: Loss=0.0000, Val Acc=100.00%
     ⏳ No improvement (1/2)


   Epoch 3/10: Loss=0.0000, Val Acc=100.00%
     ⏳ No improvement (2/2)
     🛑 EARLY STOPPING at epoch 3

  📊 FINAL ACCURACIES & FORGETTING SCORE (all 13 tasks):
    Task A (Animal vs Vehicle   ): Acc=100.00% | Peak=100.00% | FGT=0.00%
    Task B (Natural vs Man-Made ): Acc=100.00% | Peak=100.00% | FGT=0.00%
    Task C (Living vs Non-Living): Acc=100.00% | Peak=100.00% | FGT=0.00%
    Task D (Large vs Small      ): Acc=100.00% | Peak=100.00% | FGT=0.00%
    Task E (Ground vs Air/Water ): Acc=100.00% | Peak=100.00% | FGT=0.00%
    Task F (Domestic vs Wild    ): Acc=100.00% | Peak=100.00% | FGT=0.00%
    Task G (Mammal vs Non-Mammal): Acc=100.00% | Peak=100.00% | FGT=0.00%
    Task H (Flying vs Non-Flying): Acc=100.00% | Peak=100.00% | FGT=0.00%
    Task I (Fast vs Slow        ): Acc=100.00% | Peak=100.00% | FGT=0.00%
    Task J (Urban vs Rural      ): Acc=100.00% | Peak=100.00% | FGT=0.00%
    Task K (Predator vs Prey    ): Acc=100.00% | Peak=100.00% | FGT=0.00%
    Task L (Nocturnal vs

In [ ]:
!ls topo_stl10_13tasks/

topo_trained_13tasks_gemma.pt


## HF

In [ ]:
# ============================================================================
# UPLOAD TOPO-2026 GEMMA MODEL TO HUGGING FACE (FIXED)
# ============================================================================

import torch
import os
import json
import shutil
from huggingface_hub import HfApi, login
from google.colab import userdata
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. CONFIGURATION
# ============================================================================
# Retrieve token from Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')
USERNAME = 'frankmorales2020'
REPO_ID = f"{USERNAME}/topo-gemma-4-e4b-vision-13tasks"

# Local paths
LOCAL_CKPT = "./topo_stl10_13tasks/topo_trained_13tasks_gemma.pt"
MODEL_NAME = "frankmorales2020/gemma-4-e4b-unesco-optimized"
TEMP_DIR = "./temp_topo_upload"

print("="*80)
print("🚀 UPLOAD TOPO-2026 GEMMA MODEL TO HUGGING FACE")
print("="*80)

# ============================================================================
# 2. LOGIN TO HUGGING FACE
# ============================================================================
print("\n🔑 Logging in to Hugging Face...")
login(token=HF_TOKEN)
print("   ✅ Logged in successfully!")

# ============================================================================
# 3. LOAD CHECKPOINT (FIXED)
# ============================================================================
print("\n📥 Loading checkpoint...")

# FIX: Set weights_only=False to load the checkpoint
try:
    checkpoint = torch.load(LOCAL_CKPT, map_location="cpu", weights_only=False)
    print("   ✅ Loaded successfully!")
except Exception as e:
    print(f"   ❌ Error loading checkpoint: {e}")
    raise

print(f"   Best Run: {checkpoint['best_run']}")
print(f"   Best Accuracy: {checkpoint['best_avg_acc']:.2f}%")
print(f"   Boundary Layer: {checkpoint['boundary_layer']}")
print(f"   Prime Anchors: {checkpoint['prime_anchors']}")
print(f"   Safety Constant: {checkpoint['safety_constant']:.10f}")

# ============================================================================
# 4. SAVE MODEL FILES
# ============================================================================
print("\n📁 Creating model files...")

# Create temp directory
os.makedirs(TEMP_DIR, exist_ok=True)

# Save checkpoint as pytorch_model.bin (using weights_only=False for saving too)
torch.save(checkpoint, f"{TEMP_DIR}/pytorch_model.bin")
print("   ✅ Saved: pytorch_model.bin")

# Save config
config = {
    "model_type": "gemma",
    "hidden_size": checkpoint['hidden_size'],
    "vocab_size": 262144,
    "boundary_layer": checkpoint['boundary_layer'],
    "prime_anchors": checkpoint['prime_anchors'],
    "safety_constant": checkpoint['safety_constant'],
    "best_run": checkpoint['best_run'],
    "best_avg_acc": checkpoint['best_avg_acc'],
    "task_order": checkpoint['task_order'],
    "num_tasks": 13,
    "torch_dtype": "bfloat16",
    "quantization": "nf4",
    "base_model": MODEL_NAME,
}

with open(f"{TEMP_DIR}/config.json", "w") as f:
    json.dump(config, f, indent=2)
print("   ✅ Saved: config.json")

# Save tokenizer
try:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.save_pretrained(TEMP_DIR)
    print("   ✅ Saved: tokenizer files")
except Exception as e:
    print(f"   ⚠️ Could not save tokenizer: {e}")

# Create .gitattributes
with open(f"{TEMP_DIR}/.gitattributes", "w") as f:
    f.write("*.bin filter=lfs diff=lfs merge=lfs -text\n")
    f.write("*.pt filter=lfs diff=lfs merge=lfs -text\n")
    f.write("*.safetensors filter=lfs diff=lfs merge=lfs -text\n")
print("   ✅ Saved: .gitattributes")

print(f"\n   📁 Files saved to: {TEMP_DIR}")

# ============================================================================
# 5. UPLOAD TO HUGGING FACE
# ============================================================================
print(f"\n☁️ Uploading to Hugging Face...")
print(f"   Repository: {REPO_ID}")

# Initialize API
api = HfApi(token=HF_TOKEN)

# Create repository if it doesn't exist
try:
    api.create_repo(repo_id=REPO_ID, exist_ok=True, private=False)
    print("   ✅ Repository created/exists")
except Exception as e:
    print(f"   ⚠️ Repository issue: {e}")

# Upload all files
api.upload_folder(
    folder_path=TEMP_DIR,
    repo_id=REPO_ID,
    repo_type="model",
    ignore_patterns=["README.md", "README"],
)

print(f"\n✅ Model uploaded successfully!")
print(f"   🔗 https://huggingface.co/{REPO_ID}")

# ============================================================================
# 6. CLEANUP
# ============================================================================
shutil.rmtree(TEMP_DIR)
print(f"\n🧹 Cleaned up temporary files.")

# ============================================================================
# 7. VERIFICATION
# ============================================================================
print("\n📋 Verifying upload...")
try:
    from huggingface_hub import list_repo_files

    files = list_repo_files(repo_id=REPO_ID, token=HF_TOKEN)
    print("   Files in repository:")
    for f in files:
        print(f"     - {f}")
except Exception as e:
    print(f"   ⚠️ Could not verify: {e}")

print("\n" + "="*80)
print("🎉 UPLOAD COMPLETE!")
print(f"🔗 Model available at: https://huggingface.co/{REPO_ID}")
print("="*80)

🚀 UPLOAD TOPO-2026 GEMMA MODEL TO HUGGING FACE

🔑 Logging in to Hugging Face...
   ✅ Logged in successfully!

📥 Loading checkpoint...
   ✅ Loaded successfully!
   Best Run: 1
   Best Accuracy: 1.00%
   Boundary Layer: 24
   Prime Anchors: [2, 3, 5, 7, 11, 13]
   Safety Constant: 0.9785142874

📁 Creating model files...
   ✅ Saved: pytorch_model.bin
   ✅ Saved: config.json
   ✅ Saved: tokenizer files
   ✅ Saved: .gitattributes

   📁 Files saved to: ./temp_topo_upload

☁️ Uploading to Hugging Face...
   Repository: frankmorales2020/topo-gemma-4-e4b-vision-13tasks
   ✅ Repository created/exists

✅ Model uploaded successfully!
   🔗 https://huggingface.co/frankmorales2020/topo-gemma-4-e4b-vision-13tasks

🧹 Cleaned up temporary files.

📋 Verifying upload...
   Files in repository:
     - .gitattributes
     - chat_template.jinja
     - config.json
     - pytorch_model.bin
     - tokenizer.json
     - tokenizer_config.json

🎉 UPLOAD COMPLETE!
🔗 Model available at: https://huggingface.co/frankm

## INFERENCE

In [ ]:
import sys
import os
import contextlib

@contextlib.contextmanager
def suppress_all_output():
    with open(os.devnull, "w") as devnull:
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        sys.stdout = devnull
        sys.stderr = devnull
        try:
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr

os.environ["UNSLOTH_DISABLE_LOGGING"] = "1"
os.environ["TRANSVERSE_NO_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"

with suppress_all_output():
    import torch

    original_torch_load = torch.load
    def patched_torch_load(*args, **kwargs):
        kwargs["weights_only"] = False
        return original_torch_load(*args, **kwargs)
    torch.load = patched_torch_load

    from huggingface_hub import hf_hub_download
    from PIL import Image
    from unsloth import FastVisionModel

MODEL_ID = "frankmorales2020/topo-gemma-4-e4b-vision-13tasks"

with suppress_all_output():
    ckpt_path = hf_hub_download(repo_id=MODEL_ID, filename="pytorch_model.bin")
    checkpoint = torch.load(ckpt_path, map_location="cpu")
    BASE_MODEL = checkpoint.get("base_model", "frankmorales2020/gemma-4-e4b-unesco-optimized")

    model, tokenizer = FastVisionModel.from_pretrained(
        model_name=BASE_MODEL,
        load_in_4bit=True,
        dtype=torch.bfloat16,
    )
    FastVisionModel.for_inference(model)

# 1. Download image using wget and load it
image_url = "https://picsum.photos/300/300"
image_filename = "test_image.jpg"
os.system(f"wget -q -O {image_filename} {image_url}")

image = Image.open(image_filename).convert("RGB")

# 2. Define all 13 tasks
tasks = [
    ("Task A", "Animal vs Vehicle", "Does this image depict an animal or a vehicle?"),
    ("Task B", "Natural vs Man-Made", "Is this subject natural or man-made?"),
    ("Task C", "Living vs Non-Living", "Is the primary subject living or non-living?"),
    ("Task D", "Large vs Small", "Is the subject large or small in scale?"),
    ("Task E", "Ground vs Air/Water", "Does this subject belong to ground or air/water?"),
    ("Task F", "Domestic vs Wild", "Is this subject domestic or wild?"),
    ("Task G", "Mammal vs Non-Mammal", "Is this subject a mammal or non-mammal?"),
    ("Task H", "Flying vs Non-Flying", "Is this subject flying or non-flying?"),
    ("Task I", "Fast vs Slow", "Is this subject characterized as fast or slow?"),
    ("Task J", "Urban vs Rural", "Does this setting represent an urban or rural environment?"),
    ("Task K", "Predator vs Prey", "Is this subject a predator or prey?"),
    ("Task L", "Nocturnal vs Diurnal", "Is this subject nocturnal or diurnal?"),
    ("Task M", "Domesticated vs Wild Animals", "Is this animal domesticated or wild?")
]

print("\n" + "="*80)
print("🚀 EVALUATING ALL 13 TOPO-2026 TASKS")
print("="*80)

for task_id, task_name, prompt in tasks:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": f"{task_id} ({task_name}): {prompt}"}
            ]
        }
    ]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")

    with torch.inference_mode():
        output_tokens = model.generate(
            **inputs,
            max_new_tokens=24,
            do_sample=False,
            use_cache=True,
        )

    response = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    answer = response.split("model")[-1].strip() if "model" in response else response
    print(f"[{task_id}] {task_name:<30} ➔ {answer}")

print("="*80)
print("🎉 EVALUATION COMPLETE!")
print("="*80)

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]


🚀 EVALUATING ALL 13 TOPO-2026 TASKS
[Task A] Animal vs Vehicle              ➔ This image does not clearly depict an animal or a vehicle. It primarily shows a **person** reflected or visible through a
[Task B] Natural vs Man-Made            ➔ The subject in the image is a **person**, which is **natural**.

However, the image itself is a **
[Task C] Living vs Non-Living           ➔ The primary subject in the image is a **person** (a woman), which is **living**.
[Task D] Large vs Small                 ➔ Based on the image, the subject (the woman) appears to be **medium to large** in scale relative to the
[Task E] Ground vs Air/Water            ➔ Based on the image, the subject is a **person** (a woman).

In the context of the "Ground
[Task F] Domestic vs Wild               ➔ Based on the image, the subject appears to be **domestic**.

The image is a portrait of a person, and
[Task G] Mammal vs Non-Mammal           ➔ Based on the image, the subject is a **human**, and humans are **mammals

## TOPO-GEMMA-CIFAR

In [1]:
# ============================================================================
# ULTRA-FAST CIFAR-100 DATASET CREATION - STANDALONE CELL
# ============================================================================

import torch
import torchvision
import torchvision.transforms as transforms
import random
import numpy as np
from transformers import AutoTokenizer
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("⚡ ULTRA-FAST CIFAR-100 DATASET CREATION (13 TASKS)")
print("="*80)

# ============================================================================
# 1. CONFIGURATION
# ============================================================================
SEED = 123
MAX_LEN = 64
BATCH_SIZE = 8
MODEL_NAME = "frankmorales2020/gemma-4-e4b-unesco-optimized"

# ============================================================================
# 2. CIFAR-100 CLASSES
# ============================================================================
CIFAR100_SUPERCLASSES = {
    'aquatic_mammals': ['beaver', 'dolphin', 'otter', 'seal', 'whale'],
    'fish': ['aquarium_fish', 'flatfish', 'ray', 'shark', 'trout'],
    'flowers': ['orchids', 'poppies', 'roses', 'sunflowers', 'tulips'],
    'food': ['bottles', 'bowls', 'cans', 'cups', 'plates'],
    'fruit_and_vegetables': ['apples', 'mushrooms', 'oranges', 'pears', 'sweet_peppers'],
    'household_electronics': ['clock', 'computer_keyboard', 'lamp', 'telephone', 'television'],
    'household_furniture': ['bed', 'chair', 'couch', 'table', 'wardrobe'],
    'insects': ['bee', 'beetle', 'butterfly', 'caterpillar', 'cockroach'],
    'large_carnivores': ['bear', 'leopard', 'lion', 'tiger', 'wolf'],
    'large_manmade_outdoors': ['bridge', 'castle', 'house', 'road', 'skyscraper'],
    'large_natural_outdoors': ['cloud', 'forest', 'mountain', 'plain', 'sea'],
    'large_omnivores_herbivores': ['camel', 'cattle', 'chimpanzee', 'elephant', 'kangaroo'],
    'medium_mammals': ['fox', 'porcupine', 'possum', 'raccoon', 'skunk'],
    'non_insect_invertebrates': ['crab', 'lobster', 'snail', 'spider', 'worm'],
    'people': ['baby', 'boy', 'girl', 'man', 'woman'],
    'reptiles': ['crocodile', 'dinosaur', 'lizard', 'snake', 'turtle'],
    'small_mammals': ['hamster', 'mouse', 'rabbit', 'shrew', 'squirrel'],
    'trees': ['maple_tree', 'oak_tree', 'palm_tree', 'pine_tree', 'willow_tree'],
    'vehicles_1': ['bicycle', 'bus', 'motorcycle', 'pickup_truck', 'train'],
    'vehicles_2': ['lawn_mower', 'rocket', 'streetcar', 'tank', 'tractor']
}

# Map class names to indices
all_cifar_classes = []
for superclass in CIFAR100_SUPERCLASSES.values():
    all_cifar_classes.extend(superclass)

class_to_idx = {name: idx for idx, name in enumerate(all_cifar_classes)}
idx_to_class = {idx: name for name, idx in class_to_idx.items()}

print(f"\n📋 Total classes: {len(class_to_idx)}")

# ============================================================================
# 3. 13 DIVERSE TASKS ON CIFAR-100
# ============================================================================
TASKS_CIFAR13 = {
    'A': {
        'name': 'Animal vs Vehicle',
        'class0': ['beaver', 'dolphin', 'otter', 'seal', 'whale', 'camel', 'cattle',
                   'chimpanzee', 'elephant', 'kangaroo', 'bear', 'leopard', 'lion',
                   'tiger', 'wolf', 'fox', 'porcupine', 'possum', 'raccoon', 'skunk',
                   'hamster', 'mouse', 'rabbit', 'shrew', 'squirrel', 'baby', 'boy',
                   'girl', 'man', 'woman', 'crocodile', 'dinosaur', 'lizard', 'snake',
                   'turtle', 'bee', 'beetle', 'butterfly', 'caterpillar', 'cockroach',
                   'crab', 'lobster', 'snail', 'spider', 'worm', 'aquarium_fish',
                   'flatfish', 'ray', 'shark', 'trout'],
        'class1': ['bicycle', 'bus', 'motorcycle', 'pickup_truck', 'train', 'lawn_mower',
                   'rocket', 'streetcar', 'tank', 'tractor', 'bridge', 'castle', 'house',
                   'road', 'skyscraper', 'clock', 'computer_keyboard', 'lamp', 'telephone',
                   'television', 'bed', 'chair', 'couch', 'table', 'wardrobe', 'bottles',
                   'bowls', 'cans', 'cups', 'plates'],
        'label0_text': ['animal', 'living creature', 'wild animal'],
        'label1_text': ['vehicle', 'machine', 'transportation']
    },
    'B': {
        'name': 'Natural vs Man-Made',
        'class0': ['beaver', 'dolphin', 'otter', 'seal', 'whale', 'camel', 'cattle',
                   'chimpanzee', 'elephant', 'kangaroo', 'bear', 'leopard', 'lion',
                   'tiger', 'wolf', 'fox', 'porcupine', 'possum', 'raccoon', 'skunk',
                   'hamster', 'mouse', 'rabbit', 'shrew', 'squirrel', 'baby', 'boy',
                   'girl', 'man', 'woman', 'crocodile', 'dinosaur', 'lizard', 'snake',
                   'turtle', 'bee', 'beetle', 'butterfly', 'caterpillar', 'cockroach',
                   'crab', 'lobster', 'snail', 'spider', 'worm', 'aquarium_fish',
                   'flatfish', 'ray', 'shark', 'trout', 'orchids', 'poppies', 'roses',
                   'sunflowers', 'tulips', 'apples', 'mushrooms', 'oranges', 'pears',
                   'sweet_peppers', 'cloud', 'forest', 'mountain', 'plain', 'sea',
                   'maple_tree', 'oak_tree', 'palm_tree', 'pine_tree', 'willow_tree'],
        'class1': ['bicycle', 'bus', 'motorcycle', 'pickup_truck', 'train', 'lawn_mower',
                   'rocket', 'streetcar', 'tank', 'tractor', 'bridge', 'castle', 'house',
                   'road', 'skyscraper', 'clock', 'computer_keyboard', 'lamp', 'telephone',
                   'television', 'bed', 'chair', 'couch', 'table', 'wardrobe', 'bottles',
                   'bowls', 'cans', 'cups', 'plates'],
        'label0_text': ['natural', 'organic', 'from nature'],
        'label1_text': ['man-made', 'artificial', 'human-built']
    },
    'C': {
        'name': 'Living vs Non-Living',
        'class0': ['beaver', 'dolphin', 'otter', 'seal', 'whale', 'camel', 'cattle',
                   'chimpanzee', 'elephant', 'kangaroo', 'bear', 'leopard', 'lion',
                   'tiger', 'wolf', 'fox', 'porcupine', 'possum', 'raccoon', 'skunk',
                   'hamster', 'mouse', 'rabbit', 'shrew', 'squirrel', 'baby', 'boy',
                   'girl', 'man', 'woman', 'crocodile', 'dinosaur', 'lizard', 'snake',
                   'turtle', 'bee', 'beetle', 'butterfly', 'caterpillar', 'cockroach',
                   'crab', 'lobster', 'snail', 'spider', 'worm', 'aquarium_fish',
                   'flatfish', 'ray', 'shark', 'trout', 'orchids', 'poppies', 'roses',
                   'sunflowers', 'tulips', 'apples', 'mushrooms', 'oranges', 'pears',
                   'sweet_peppers', 'maple_tree', 'oak_tree', 'palm_tree', 'pine_tree',
                   'willow_tree'],
        'class1': ['bicycle', 'bus', 'motorcycle', 'pickup_truck', 'train', 'lawn_mower',
                   'rocket', 'streetcar', 'tank', 'tractor', 'bridge', 'castle', 'house',
                   'road', 'skyscraper', 'clock', 'computer_keyboard', 'lamp', 'telephone',
                   'television', 'bed', 'chair', 'couch', 'table', 'wardrobe', 'bottles',
                   'bowls', 'cans', 'cups', 'plates', 'cloud', 'forest', 'mountain',
                   'plain', 'sea'],
        'label0_text': ['living', 'alive', 'breathing'],
        'label1_text': ['non-living', 'inanimate', 'not alive']
    },
    'D': {
        'name': 'Large vs Small',
        'class0': ['bear', 'leopard', 'lion', 'tiger', 'wolf', 'camel', 'cattle',
                   'chimpanzee', 'elephant', 'kangaroo', 'dolphin', 'whale', 'seal'],
        'class1': ['fox', 'porcupine', 'possum', 'raccoon', 'skunk', 'hamster', 'mouse',
                   'rabbit', 'shrew', 'squirrel', 'bee', 'beetle', 'butterfly',
                   'caterpillar', 'cockroach', 'crab', 'lobster', 'snail', 'spider', 'worm'],
        'label0_text': ['large', 'big', 'large-sized'],
        'label1_text': ['small', 'tiny', 'small-sized']
    },
    'E': {
        'name': 'Ground vs Air/Water',
        'class0': ['bear', 'leopard', 'lion', 'tiger', 'wolf', 'camel', 'cattle',
                   'chimpanzee', 'elephant', 'kangaroo', 'fox', 'porcupine', 'possum',
                   'raccoon', 'skunk', 'hamster', 'mouse', 'rabbit', 'shrew', 'squirrel',
                   'baby', 'boy', 'girl', 'man', 'woman'],
        'class1': ['beaver', 'dolphin', 'otter', 'seal', 'whale', 'aquarium_fish',
                   'flatfish', 'ray', 'shark', 'trout', 'bee', 'beetle', 'butterfly',
                   'caterpillar', 'cockroach'],
        'label0_text': ['ground', 'land-based', 'terrestrial'],
        'label1_text': ['air or water', 'non-terrestrial', 'flying/swimming']
    },
    'F': {
        'name': 'Domestic vs Wild',
        'class0': ['cattle', 'hamster', 'mouse', 'rabbit', 'shrew'],
        'class1': ['bear', 'leopard', 'lion', 'tiger', 'wolf', 'fox', 'porcupine',
                   'possum', 'raccoon', 'skunk', 'crocodile', 'dinosaur', 'lizard',
                   'snake', 'turtle'],
        'label0_text': ['domestic', 'tame', 'pet'],
        'label1_text': ['wild', 'untamed', 'savage']
    },
    'G': {
        'name': 'Mammal vs Non-Mammal',
        'class0': ['beaver', 'dolphin', 'otter', 'seal', 'whale', 'camel', 'cattle',
                   'chimpanzee', 'elephant', 'kangaroo', 'bear', 'leopard', 'lion',
                   'tiger', 'wolf', 'fox', 'porcupine', 'possum', 'raccoon', 'skunk',
                   'hamster', 'mouse', 'rabbit', 'shrew', 'squirrel', 'baby', 'boy',
                   'girl', 'man', 'woman'],
        'class1': ['crocodile', 'dinosaur', 'lizard', 'snake', 'turtle', 'bee', 'beetle',
                   'butterfly', 'caterpillar', 'cockroach', 'crab', 'lobster', 'snail',
                   'spider', 'worm', 'aquarium_fish', 'flatfish', 'ray', 'shark', 'trout',
                   'orchids', 'poppies', 'roses', 'sunflowers', 'tulips', 'apples',
                   'mushrooms', 'oranges', 'pears', 'sweet_peppers', 'maple_tree',
                   'oak_tree', 'palm_tree', 'pine_tree', 'willow_tree', 'bicycle', 'bus',
                   'motorcycle', 'pickup_truck', 'train', 'lawn_mower', 'rocket',
                   'streetcar', 'tank', 'tractor'],
        'label0_text': ['mammal', 'warm-blooded', 'fur-bearing'],
        'label1_text': ['non-mammal', 'cold-blooded', 'feathered/metal']
    },
    'H': {
        'name': 'Flying vs Non-Flying',
        'class0': ['bee', 'beetle', 'butterfly', 'caterpillar', 'cockroach'],
        'class1': ['bear', 'leopard', 'lion', 'tiger', 'wolf', 'camel', 'cattle',
                   'chimpanzee', 'elephant', 'kangaroo', 'fox', 'porcupine', 'possum',
                   'raccoon', 'skunk', 'hamster', 'mouse', 'rabbit', 'shrew', 'squirrel',
                   'crocodile', 'dinosaur', 'lizard', 'snake', 'turtle', 'crab', 'lobster',
                   'snail', 'spider', 'worm', 'bicycle', 'bus', 'motorcycle', 'pickup_truck',
                   'train', 'lawn_mower', 'rocket', 'streetcar', 'tank', 'tractor'],
        'label0_text': ['flying', 'can fly', 'airborne'],
        'label1_text': ['non-flying', 'ground-based', 'earthbound']
    },
    'I': {
        'name': 'Fast vs Slow',
        'class0': ['bear', 'leopard', 'lion', 'tiger', 'wolf', 'dolphin', 'whale', 'shark'],
        'class1': ['beaver', 'otter', 'seal', 'camel', 'cattle', 'chimpanzee', 'elephant',
                   'kangaroo', 'fox', 'porcupine', 'possum', 'raccoon', 'skunk', 'hamster',
                   'mouse', 'rabbit', 'shrew', 'squirrel', 'turtle', 'snail', 'worm'],
        'label0_text': ['fast-moving', 'quick', 'rapid'],
        'label1_text': ['slow-moving', 'slow', 'lethargic']
    },
    'J': {
        'name': 'Urban vs Rural',
        'class0': ['bicycle', 'bus', 'motorcycle', 'pickup_truck', 'train', 'lawn_mower',
                   'rocket', 'streetcar', 'tank', 'tractor', 'bridge', 'castle', 'house',
                   'road', 'skyscraper', 'clock', 'computer_keyboard', 'lamp', 'telephone',
                   'television', 'bed', 'chair', 'couch', 'table', 'wardrobe'],
        'class1': ['beaver', 'dolphin', 'otter', 'seal', 'whale', 'camel', 'cattle',
                   'chimpanzee', 'elephant', 'kangaroo', 'bear', 'leopard', 'lion',
                   'tiger', 'wolf', 'fox', 'porcupine', 'possum', 'raccoon', 'skunk',
                   'hamster', 'mouse', 'rabbit', 'shrew', 'squirrel', 'cloud', 'forest',
                   'mountain', 'plain', 'sea', 'maple_tree', 'oak_tree', 'palm_tree',
                   'pine_tree', 'willow_tree', 'orchids', 'poppies', 'roses', 'sunflowers',
                   'tulips', 'apples', 'mushrooms', 'oranges', 'pears', 'sweet_peppers'],
        'label0_text': ['urban', 'city', 'man-made environment'],
        'label1_text': ['rural', 'countryside', 'natural environment']
    },
    'K': {
        'name': 'Predator vs Prey',
        'class0': ['bear', 'leopard', 'lion', 'tiger', 'wolf', 'shark', 'crocodile'],
        'class1': ['beaver', 'dolphin', 'otter', 'seal', 'whale', 'camel', 'cattle',
                   'chimpanzee', 'elephant', 'kangaroo', 'fox', 'porcupine', 'possum',
                   'raccoon', 'skunk', 'hamster', 'mouse', 'rabbit', 'shrew', 'squirrel'],
        'label0_text': ['predator', 'hunter', 'carnivore'],
        'label1_text': ['prey', 'herbivore', 'hunted']
    },
    'L': {
        'name': 'Nocturnal vs Diurnal',
        'class0': ['bear', 'leopard', 'lion', 'tiger', 'wolf', 'fox', 'raccoon', 'skunk'],
        'class1': ['beaver', 'dolphin', 'otter', 'seal', 'whale', 'camel', 'cattle',
                   'chimpanzee', 'elephant', 'kangaroo', 'porcupine', 'possum', 'hamster',
                   'mouse', 'rabbit', 'shrew', 'squirrel', 'baby', 'boy', 'girl', 'man',
                   'woman'],
        'label0_text': ['nocturnal', 'night-active', 'night'],
        'label1_text': ['diurnal', 'day-active', 'day']
    },
    'M': {
        'name': 'Domesticated vs Wild Animals',
        'class0': ['cattle', 'hamster', 'mouse', 'rabbit', 'shrew'],
        'class1': ['bear', 'leopard', 'lion', 'tiger', 'wolf', 'fox', 'porcupine',
                   'possum', 'raccoon', 'skunk', 'crocodile', 'dinosaur', 'lizard',
                   'snake', 'turtle', 'bee', 'beetle', 'butterfly', 'caterpillar',
                   'cockroach', 'crab', 'lobster', 'snail', 'spider', 'worm'],
        'label0_text': ['domesticated', 'pet', 'tame animal'],
        'label1_text': ['wild animal', 'untamed', 'free']
    },
}

TASK_ORDER = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M']

print(f"\n📌 13 DIVERSE TASKS:")
for task_id in TASK_ORDER:
    task = TASKS_CIFAR13[task_id]
    print(f"   {task_id}: {task['name']}")

# ============================================================================
# 4. LOAD CIFAR-100
# ============================================================================
print(f"\n📚 LOADING CIFAR-100...")
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR100(
    root='./data', train=True, download=True, transform=transform
)
testset = torchvision.datasets.CIFAR100(
    root='./data', train=False, download=True, transform=transform
)

print(f"   Training set: {len(trainset):,} samples")
print(f"   Test set: {len(testset):,} samples")

# ============================================================================
# 5. LOAD TOKENIZER
# ============================================================================
print(f"\n👁️ Loading Tokenizer...")
try:
    import contextlib
    import io
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        from unsloth import FastVisionModel
        _, vision_processor = FastVisionModel.from_pretrained(
            MODEL_NAME,
            load_in_4bit=True,
            dtype=torch.bfloat16,
            device_map="auto",
        )
    tokenizer = vision_processor.tokenizer if hasattr(vision_processor, 'tokenizer') else vision_processor
    print("✅ Tokenizer loaded")
except:
    tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b", trust_remote_code=True)
    print("⚠️ Using fallback tokenizer")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ============================================================================
# 6. ULTRA-FAST: BUILD MASTER INDEX (ONCE!)
# ============================================================================
print(f"\n⚡ Building master class indices (SINGLE PASS)...")

master_train_indices = {}
for idx, (_, label) in enumerate(tqdm(trainset, desc="   Training set")):
    cls = idx_to_class[label]
    if cls not in master_train_indices:
        master_train_indices[cls] = []
    master_train_indices[cls].append(idx)

master_test_indices = {}
for idx, (_, label) in enumerate(tqdm(testset, desc="   Test set")):
    cls = idx_to_class[label]
    if cls not in master_test_indices:
        master_test_indices[cls] = []
    master_test_indices[cls].append(idx)

print(f"   ✅ Built indices for {len(master_train_indices)} training classes")
print(f"   ✅ Built indices for {len(master_test_indices)} test classes")

# ============================================================================
# 7. HELPER FUNCTIONS
# ============================================================================
def get_class_label(cls, task):
    if cls in task['class0']:
        return 0
    else:
        return 1

def create_vision_text(cls, task_type):
    task = TASKS_CIFAR13[task_type]
    if cls in task['class0']:
        prefixes = [f"A {cls} is {t}" for t in task['label0_text']]
        prefixes += [f"A {t} {cls}" for t in task['label0_text']]
    else:
        prefixes = [f"A {cls} is {t}" for t in task['label1_text']]
        prefixes += [f"A {t} {cls}" for t in task['label1_text']]
    return random.choice(prefixes)

def create_dataset_fast(class_list, num_samples, master_indices, task_type):
    """ULTRA-FAST: Uses pre-built master indices"""
    random.seed(SEED)
    texts, labels = [], []
    samples_per_class = num_samples // len(class_list)

    for cls in class_list:
        indices = master_indices.get(cls, [])
        available = min(len(indices), samples_per_class * 3)
        if available == 0:
            continue
        selected = random.sample(indices, min(available, num_samples))
        for idx in selected:
            texts.append(create_vision_text(cls, task_type))
            labels.append(get_class_label(cls, TASKS_CIFAR13[task_type]))

    return texts, labels

# ============================================================================
# 8. CREATE ALL DATASETS (ULTRA-FAST!)
# ============================================================================
num_samples = 2000
task_loaders = {}
test_loaders = {}

print(f"\n📚 Creating 13 task datasets (ULTRA-FAST - seconds per task)...")

for task_id in tqdm(TASK_ORDER, desc="   Tasks"):
    task = TASKS_CIFAR13[task_id]
    class_list = task['class0'] + task['class1']

    # Training set
    texts, labels = create_dataset_fast(class_list, num_samples, master_train_indices, task_id)

    tokens = tokenizer(texts, max_length=MAX_LEN, padding='max_length',
                       truncation=True, return_tensors='pt')

    dataset = torch.utils.data.TensorDataset(
        tokens.input_ids,
        tokens.attention_mask,
        torch.tensor(labels, dtype=torch.long)
    )

    loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    task_loaders[task_id] = loader

    # Test set
    test_texts, test_labels = create_dataset_fast(class_list, 400, master_test_indices, task_id)
    test_tokens = tokenizer(test_texts, max_length=MAX_LEN, padding='max_length',
                           truncation=True, return_tensors='pt')

    test_dataset = torch.utils.data.TensorDataset(
        test_tokens.input_ids,
        test_tokens.attention_mask,
        torch.tensor(test_labels, dtype=torch.long)
    )

    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loaders[task_id] = test_loader

# ============================================================================
# 9. SUMMARY
# ============================================================================
print(f"\n" + "="*80)
print(f"✅ DATASET CREATION COMPLETE!")
print("="*80)

print(f"\n📊 Summary:")
total_train = 0
total_test = 0
for task_id in TASK_ORDER:
    train_size = len(task_loaders[task_id].dataset)
    test_size = len(test_loaders[task_id].dataset)
    total_train += train_size
    total_test += test_size
    print(f"   Task {task_id}: {train_size} train, {test_size} test")

print(f"\n   Total: {total_train:,} training samples, {total_test:,} test samples")
print(f"   Tasks: {len(TASK_ORDER)}")
print(f"   Classes: {len(class_to_idx)}")

print("\n" + "="*80)
print("🎉 READY FOR TRAINING!")
print("="*80)

# Store datasets for use in the main script
# These variables are now available: task_loaders, test_loaders, TASK_ORDER, TASKS_CIFAR13

⚡ ULTRA-FAST CIFAR-100 DATASET CREATION (13 TASKS)

📋 Total classes: 100

📌 13 DIVERSE TASKS:
   A: Animal vs Vehicle
   B: Natural vs Man-Made
   C: Living vs Non-Living
   D: Large vs Small
   E: Ground vs Air/Water
   F: Domestic vs Wild
   G: Mammal vs Non-Mammal
   H: Flying vs Non-Flying
   I: Fast vs Slow
   J: Urban vs Rural
   K: Predator vs Prey
   L: Nocturnal vs Diurnal
   M: Domesticated vs Wild Animals

📚 LOADING CIFAR-100...
   Training set: 50,000 samples
   Test set: 10,000 samples

👁️ Loading Tokenizer...


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Gemma4ForConditionalGeneration LOAD REPORT from: frankmorales2020/gemma-4-e4b-unesco-optimized
Key                                                     | Status     |  | 
--------------------------------------------------------+------------+--+-
language_model.layers.{24...41}.self_attn.v_proj.weight | UNEXPECTED |  | 
language_model.layers.{24...41}.self_attn.k_norm.weight | UNEXPECTED |  | 
language_model.layers.{24...41}.self_attn.k_proj.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Tokenizer loaded

⚡ Building master class indices (SINGLE PASS)...


   Test set: 100%|██████████| 10000/10000 [00:02<00:00, 4162.07it/s]


   ✅ Built indices for 100 training classes
   ✅ Built indices for 100 test classes

📚 Creating 13 task datasets (ULTRA-FAST - seconds per task)...


   Tasks: 100%|██████████| 13/13 [00:06<00:00,  1.92it/s]


✅ DATASET CREATION COMPLETE!

📊 Summary:
   Task A: 6000 train, 1200 test
   Task B: 6000 train, 1200 test
   Task C: 6000 train, 1200 test
   Task D: 5940 train, 1188 test
   Task E: 6000 train, 1200 test
   Task F: 6000 train, 1200 test
   Task G: 5850 train, 1125 test
   Task H: 5940 train, 1080 test
   Task I: 5916 train, 1131 test
   Task J: 5880 train, 1050 test
   Task K: 5994 train, 1134 test
   Task L: 5940 train, 1170 test
   Task M: 5940 train, 1170 test

   Total: 77,400 training samples, 15,048 test samples
   Tasks: 13
   Classes: 100

🎉 READY FOR TRAINING!


TRAINING

In [ ]:
# ============================================================================
# TOPO-2026: CIFAR-100 TRAINING - 5 RUNS (USES PRE-BUILT DATASETS)
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import gc
import random
import os
import json
import time
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("🔬 TOPO-2026: CIFAR-100 TRAINING - 5 RUNS")
print("   USING PRE-BUILT ULTRA-FAST DATASETS")
print("="*80)

# ============================================================================
# 1. VERIFY DATASETS EXIST
# ============================================================================
try:
    # Check if datasets were created
    assert 'task_loaders' in globals() and 'test_loaders' in globals()
    assert len(task_loaders) == 13
    print(f"\n✅ Datasets found: {len(task_loaders)} tasks ready")
except:
    print("\n❌ ERROR: Datasets not found!")
    print("   Please run the ULTRA-FAST dataset creation cell first.")
    print("   Or run the standalone dataset creator.")
    raise SystemExit

# ============================================================================
# 2. CONFIGURATION
# ============================================================================
SEED = 123
N_RUNS = 5
BATCH_SIZE = 8
MAX_EPOCHS = 10
PATIENCE = 2
MAX_LEN = 64
NUM_TASKS = 13
BOUNDARY_LAYER = 24

MODEL_NAME = "frankmorales2020/gemma-4-e4b-unesco-optimized"
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]
SAFETY_CONSTANT = 1.0 - np.prod([1.0 - (p ** -0.5) for p in PRIME_ANCHORS])

LR_GRID = [
    (5e-3, 1e-3),   # Run 1
    (1e-3, 5e-4),   # Run 2
    (5e-3, 5e-3),   # Run 3
    (2e-3, 1e-3),   # Run 4
    (1e-3, 1e-3),   # Run 5
]

print(f"\n📋 Configuration:")
print(f"   Dataset: CIFAR-100")
print(f"   Tasks: {NUM_TASKS}")
print(f"   Runs: {N_RUNS}")
print(f"   Boundary Layer: {BOUNDARY_LAYER}")
print(f"   LR Grid:")
for i, (lr_embed, lr_cls) in enumerate(LR_GRID):
    print(f"      Run {i+1}: lr_embed={lr_embed:.0e}, lr_cls={lr_cls:.0e}")

# ============================================================================
# 3. LOAD VISION MODEL
# ============================================================================
print(f"\n👁️ Loading Vision Model: Gemma-4-E4B...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"   Device: {device}")

import contextlib
import io

vision_model = None
vision_processor = None

try:
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        from unsloth import FastVisionModel
        vision_model, vision_processor = FastVisionModel.from_pretrained(
            MODEL_NAME,
            load_in_4bit=True,
            dtype=torch.bfloat16,
            device_map="auto",
        )
        FastVisionModel.for_inference(vision_model)
    print("✅ Gemma Loaded (Unsloth)")
except Exception as e:
    print(f"⚠️ Unsloth failed: {e}")
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        vision_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True
        )
        print("✅ Gemma Loaded (Transformers)")
    except Exception as e2:
        print(f"⚠️ Gemma failed: {e2}")
        vision_model = None
        raise

hidden_size = 2560

# ============================================================================
# 4. CLASSIFIER MODEL - 13 HEADS
# ============================================================================
class GemmaVisionClassifier13(nn.Module):
    def __init__(self, vision_model, hidden_size=2560, boundary_layer=BOUNDARY_LAYER):
        super().__init__()
        self.vision_model = vision_model
        self.hidden_size = hidden_size
        self.boundary_layer = boundary_layer

        for task_id in TASK_ORDER:
            setattr(self, f'classifier_{task_id}', nn.Linear(hidden_size, 2))

        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs = self.vision_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )

        if hasattr(outputs, 'hidden_states') and outputs.hidden_states is not None:
            if len(outputs.hidden_states) > self.boundary_layer:
                hidden_states = outputs.hidden_states[self.boundary_layer]
            else:
                hidden_states = outputs.hidden_states[-1]
        else:
            hidden_states = outputs.last_hidden_state

        hidden_states = hidden_states.float()

        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)
        else:
            pooled = hidden_states.mean(dim=1)

        head = getattr(self, f'classifier_{self.current_task}')
        return head(pooled)

    def switch_task(self, task):
        assert task in TASK_ORDER
        self.current_task = task

    def freeze_previous_heads(self, task):
        task_idx = TASK_ORDER.index(task)
        for i in range(task_idx):
            prev_task = TASK_ORDER[i]
            head = getattr(self, f'classifier_{prev_task}')
            for param in head.parameters():
                param.requires_grad = False

# ============================================================================
# 5. TOPOLOGICAL GOVERNOR
# ============================================================================
class TopologicalGovernor:
    def __init__(self, model: nn.Module, boundary_layer=BOUNDARY_LAYER):
        self.model = model
        self.boundary_layer = boundary_layer
        self.reference_anchors = {}
        self.safety_constant = SAFETY_CONSTANT
        self.snapshot = {}
        self._register_topo_anchors()

    def _register_topo_anchors(self):
        print(f"   Initializing TOPO-2026 Topological Governor anchor snapshots (Boundary Layer: {self.boundary_layer})...")
        count = 0
        with torch.no_grad():
            for name, param in self.model.named_parameters():
                if (param.is_floating_point() or param.is_complex()) and param.ndim >= 1:
                    if f"layers.{self.boundary_layer}" in name or f"blocks.{self.boundary_layer}" in name or any(f"layer.{b}" in name for b in [23, 24, 25]):
                        snapshot = {}
                        for p in PRIME_ANCHORS:
                            if p < param.shape[0]:
                                snapshot[p] = param.data[p].clone()
                        if snapshot:
                            self.reference_anchors[name] = snapshot
                            count += 1
        print(f"   Topological Governor successfully locked prime reference anchors across {count} tensors at Boundary Layer {self.boundary_layer}.")

    def take_snapshot(self):
        embed_layer = self.model.vision_model.get_input_embeddings()
        vocab_size = embed_layer.weight.shape[0]
        anchor_indices = [p for p in PRIME_ANCHORS if p < vocab_size]
        self.snapshot = {idx: embed_layer.weight[idx].detach().clone().float() for idx in anchor_indices}
        self._register_topo_anchors()

    @torch.no_grad()
    def enforce_anchors(self):
        if not self.reference_anchors:
            return
        with torch.no_grad():
            for name, param in self.model.named_parameters():
                if name in self.reference_anchors:
                    dtype = param.dtype
                    for p, val in self.reference_anchors[name].items():
                        if p < param.shape[0]:
                            param.data[p].copy_(val.to(dtype=dtype))

    @torch.no_grad()
    def zero_anchor_gradients(self):
        with torch.no_grad():
            for name, param in self.model.named_parameters():
                if param.requires_grad and param.ndim >= 1 and param.grad is not None:
                    if name in self.reference_anchors:
                        for p in self.reference_anchors[name].keys():
                            if p < param.grad.shape[0]:
                                param.grad[p] = 0.0

    def verify_integrity(self, atol: float = 1e-5) -> bool:
        if not self.reference_anchors:
            return True
        with torch.no_grad():
            for name, param in self.model.named_parameters():
                if name in self.reference_anchors:
                    for p, val in self.reference_anchors[name].items():
                        if p < param.shape[0]:
                            if not torch.allclose(param.data[p].float(), val.float(), atol=atol):
                                return False
        return True

# ============================================================================
# 6. TRAINING FUNCTIONS
# ============================================================================
def train_task(task_label, model, loader, governor, lr_embed, lr_cls, max_epochs, patience):
    model.switch_task(task_label)
    model.train()

    head = getattr(model, f'classifier_{task_label}')
    embed_layer = model.vision_model.get_input_embeddings()

    optimizer = torch.optim.AdamW([
        {'params': embed_layer.parameters(), 'lr': lr_embed, 'weight_decay': 1e-4},
        {'params': head.parameters(), 'lr': lr_cls, 'weight_decay': 1e-4},
    ])

    best_acc = 0.0
    patience_counter = 0
    best_classifier_state = None

    for epoch in range(max_epochs):
        epoch_loss = 0
        num_batches = 0

        for input_ids, attention_mask, labels in tqdm(loader, desc=f"   Epoch {epoch+1}/{max_epochs}", leave=False):
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss = F.cross_entropy(logits, labels)
            loss.backward()

            if governor:
                governor.zero_anchor_gradients()

            torch.nn.utils.clip_grad_norm_(embed_layer.parameters(), max_norm=1.0)
            optimizer.step()

            if governor:
                governor.enforce_anchors()

            epoch_loss += loss.item()
            num_batches += 1

        avg_loss = epoch_loss / num_batches if num_batches > 0 else 0
        val_acc = evaluate_model(model, test_loaders[task_label], task_label)

        print(f"   Epoch {epoch+1}/{max_epochs}: Loss={avg_loss:.4f}, Val Acc={val_acc*100:.2f}%")

        if val_acc > best_acc:
            best_acc = val_acc
            patience_counter = 0
            best_classifier_state = head.state_dict()
            print(f"     ✅ New best: {best_acc*100:.2f}%")
        else:
            patience_counter += 1
            print(f"     ⏳ No improvement ({patience_counter}/{patience})")

        if patience_counter >= patience and epoch > 1:
            print(f"     🛑 EARLY STOPPING at epoch {epoch+1}")
            if best_classifier_state is not None:
                head.load_state_dict(best_classifier_state)
            break

    if best_classifier_state is not None:
        head.load_state_dict(best_classifier_state)

@torch.no_grad()
def evaluate_model(model, loader, task):
    if loader is None:
        return 0.0
    model.switch_task(task)
    model.eval()

    all_preds, all_labels = [], []
    for input_ids, attention_mask, labels in loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)
        logits = model(input_ids, attention_mask)
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return accuracy_score(all_labels, all_preds) if len(all_labels) > 0 else 0.0

def convert_to_serializable(obj):
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.bool_): return bool(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, dict): return {k: convert_to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list): return [convert_to_serializable(i) for i in obj]
    return obj

# ============================================================================
# 7. MAIN TRAINING LOOP - 5 RUNS
# ============================================================================
print(f"\n" + "="*80)
print(f"🚀 STARTING {N_RUNS}-RUN TRAINING (13 TASKS ON CIFAR-100)")
print("="*80)

# Create save directory
SAVE_DIR = "./topo_cifar100_13tasks"
os.makedirs(SAVE_DIR, exist_ok=True)

all_results = []
best_run = None
global_best_avg_acc = 0.0
global_best_model_state = None

for run_id in range(N_RUNS):
    torch.manual_seed(SEED + run_id)
    np.random.seed(SEED + run_id)
    random.seed(SEED + run_id)

    lr_embed, lr_cls = LR_GRID[run_id]

    print(f"\n  {'═'*80}")
    print(f"  RUN {run_id + 1}/{N_RUNS}  |  lr_embed={lr_embed:.0e}  lr_cls={lr_cls:.0e}")
    print(f"  {'═'*80}")

    # Initialize model
    model = GemmaVisionClassifier13(vision_model, hidden_size, boundary_layer=BOUNDARY_LAYER).to(device)
    embed_layer = model.vision_model.get_input_embeddings()
    embed_layer.weight.requires_grad = True

    # Zero-shot evaluation
    print(f"\n  [ZERO-SHOT] Evaluating tasks...")
    zero_accs = {}
    for task_id in TASK_ORDER[:5]:
        zero_accs[task_id] = evaluate_model(model, test_loaders[task_id], task_id)
    zero_str = ", ".join([f"{k}={zero_accs[k]*100:.2f}%" for k in zero_accs])
    print(f"    Zero-shot (first 5): {zero_str}")

    # Initialize Governor
    governor = TopologicalGovernor(model, boundary_layer=BOUNDARY_LAYER)
    governor.take_snapshot()
    print(f"  🔒 Safety Constant Λ: {governor.safety_constant:.10f}")

    task_peak_accs = {t: 0.0 for t in TASK_ORDER}

    # Train each task
    for task_idx, task_id in enumerate(TASK_ORDER):
        print(f"\n  📚 TASK {task_id}: {TASKS_CIFAR13[task_id]['name']}")

        if task_idx > 0:
            model.freeze_previous_heads(task_id)

        train_task(task_id, model, task_loaders[task_id], governor,
                   lr_embed, lr_cls, MAX_EPOCHS, PATIENCE)

        # Evaluate all tasks so far
        for past_idx in range(task_idx + 1):
            past_task = TASK_ORDER[past_idx]
            curr_acc = evaluate_model(model, test_loaders[past_task], past_task)
            if curr_acc > task_peak_accs[past_task]:
                task_peak_accs[past_task] = curr_acc

        # Verify topological integrity
        assert governor.verify_integrity(), f"❌ Topological integrity violated at Task {task_id}!"

    # Final results
    print(f"\n  📊 FINAL ACCURACIES & FORGETTING SCORE (all 13 tasks):")
    final_accs = {}
    task_forgetting = {}

    for task_id in TASK_ORDER:
        acc = evaluate_model(model, test_loaders[task_id], task_id)
        final_accs[task_id] = acc
        peak = task_peak_accs[task_id]
        fgt = max(0.0, peak - acc)
        task_forgetting[task_id] = fgt
        print(f"    Task {task_id} ({TASKS_CIFAR13[task_id]['name'][:20]:20}): Acc={acc*100:.2f}% | Peak={peak*100:.2f}% | FGT={fgt*100:.2f}%")

    global_fgt = np.mean(list(task_forgetting.values()))
    print(f"\n  📉 Global Average Forgetting Score (FGT) for Run {run_id + 1}: {global_fgt*100:.4f}%")

    all_perfect = all(acc == 1.0 for acc in final_accs.values())
    if all_perfect:
        print(f"  🎉🎉🎉 ALL 13 TASKS AT 100%! 🎉🎉🎉")

    avg_acc = np.mean(list(final_accs.values()))

    # Save best model
    if avg_acc > global_best_avg_acc:
        global_best_avg_acc = avg_acc
        global_best_model_state = {
            t: getattr(model, f'classifier_{t}').state_dict()
            for t in TASK_ORDER
        }
        best_run = run_id

    run_result = {
        'run_id': run_id,
        'lr_embed': lr_embed,
        'lr_cls': lr_cls,
        'all_perfect': all_perfect,
        'avg_accuracy': float(avg_acc * 100),
        'global_forgetting': float(global_fgt * 100),
        'final_accs': {k: float(v * 100) for k, v in final_accs.items()},
        'peak_accs': {k: float(v * 100) for k, v in task_peak_accs.items()},
        'forgetting': {k: float(v * 100) for k, v in task_forgetting.items()},
    }
    all_results.append(run_result)

    # Cleanup
    del model
    gc.collect()
    torch.cuda.empty_cache()

# ============================================================================
# 8. SAVE MODEL TO DISK
# ============================================================================
print(f"\n" + "="*80)
print(f"💾 SAVING MODEL TO LOCAL DISK")
print("="*80)

embed_layer = vision_model.get_input_embeddings()
embed_w = embed_layer.weight.detach().cpu().float()

torch.save({
    'classifiers': global_best_model_state,
    'embed_tokens_weight': embed_w,
    'prime_anchors': PRIME_ANCHORS,
    'boundary_layer': BOUNDARY_LAYER,
    'safety_constant': SAFETY_CONSTANT,
    'hidden_size': hidden_size,
    'seed': SEED,
    'runs': N_RUNS,
    'task_order': TASK_ORDER,
    'task_definitions': TASKS_CIFAR13,
    'best_run': best_run + 1 if best_run is not None else 0,
    'best_avg_acc': float(global_best_avg_acc),
    'dataset': 'CIFAR-100',
    'num_tasks': NUM_TASKS,
}, f"{SAVE_DIR}/topo_trained_13tasks_cifar100_gemma.pt")

print(f"   ✅ Saved: {SAVE_DIR}/topo_trained_13tasks_cifar100_gemma.pt")

# ============================================================================
# 9. SAVE EVALUATION METRICS
# ============================================================================
eval_data = {
    'model': 'google/gemma-4-E4B-it',
    'dataset': 'CIFAR-100',
    'num_tasks': NUM_TASKS,
    'runs': N_RUNS,
    'evaluation_date': time.strftime("%Y-%m-%d %H:%M:%S"),
    'task_order': TASK_ORDER,
    'task_definitions': {k: {'name': v['name']} for k, v in TASKS_CIFAR13.items()},
    'lr_grid': [{'run': i+1, 'lr_embed': lr_embed, 'lr_cls': lr_cls} for i, (lr_embed, lr_cls) in enumerate(LR_GRID)],
    'results': all_results,
    'best_run': best_run + 1 if best_run is not None else 0,
    'best_avg_accuracy': float(global_best_avg_acc),
}

with open(f"{SAVE_DIR}/evaluation_metrics_cifar100.json", "w") as f:
    json.dump(convert_to_serializable(eval_data), f, indent=2)

print(f"   ✅ Saved: {SAVE_DIR}/evaluation_metrics_cifar100.json")

# ============================================================================
# 10. TOPO PROTOCOL RESULTS
# ============================================================================
print(f"\n" + "="*80)
print(f"📊 TOPO PROTOCOL RESULTS SUMMARY (CIFAR-100, 13 TASKS)")
print("="*80)

print(f"\n  {'Run':<10} {'Last Task Acc':<20} {'FGT':<15} {'Avg Acc':<15} {'TOPO Result'}")
print(f"  {'-'*80}")

for i, r in enumerate(all_results):
    last_task = TASK_ORDER[-1]
    last_acc = r['final_accs'][last_task]
    fgt = r['global_forgetting']
    avg_acc = r['avg_accuracy']

    if last_acc > 85 and fgt <= 10:
        result = "✅ PASSED"
    else:
        result = "❌ FAILED"

    print(f"  Run {i+1:<6} {last_acc:<20.2f} {fgt:<15.4f} {avg_acc:<15.2f} {result}")

print(f"\n  {'═'*80}")
passed = sum(1 for r in all_results if r['final_accs'][TASK_ORDER[-1]] > 85 and r['global_forgetting'] <= 10)
print(f"  Runs: {N_RUNS}")
print(f"  Passed: {passed}/{N_RUNS} ({passed/N_RUNS*100:.1f}%)")
print(f"  Failed: {N_RUNS - passed}/{N_RUNS}")

if passed == N_RUNS:
    print(f"\n  🎉🎉🎉 ALL {N_RUNS} RUNS PASSED TOPO PROTOCOL ON CIFAR-100! 🎉🎉🎉")
    print(f"\n  ✅ Catastrophic Forgetting is SOLVED on CIFAR-100 with 13 diverse tasks!")
    print(f"\n  📊 Best Run: Run {best_run + 1}")
    print(f"     Avg Accuracy: {global_best_avg_acc:.2f}%")
    print(f"     Best FGT: {min([r['global_forgetting'] for r in all_results]):.4f}%")
else:
    print(f"\n  ⚠️ Some runs failed the protocol")

print("\n" + "="*80)
print("🎉 COMPLETE! TOPOLOGICAL GOVERNOR PROVEN ON CIFAR-100")
print("="*80)

# ============================================================================
# 11. LIST SAVED FILES
# ============================================================================
print(f"\n📁 Saved files in {SAVE_DIR}:")
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / (1024 * 1024)
    print(f"   - {f} ({size:.2f} MB)")